# DMLC Fluence 演算法教學 Notebook

本 notebook 示範兩種動態多葉準直儀（DMLC）的 fluence 生成演算法：

1. **Sliding Window**（Convery-Rosenbloom / Spirou-Chui）— 葉片持續移動，`BeamType = DYNAMIC`
2. **Step-and-Shoot**（Bortfeld / Engel 分解）— 葉片在 segment 間靜止，`BeamType = STATIC`

輸入：
- `target_fluence.png`：目標 fluence 圖
- `demo.dcm`：原始 RT Plan，作為 DICOM 模板（提供 MLC 幾何、機型資訊、isocenter 等）

輸出：
- `output_sliding.dcm` / `output_stepshoot.dcm`：可直接匯入 TPS 驗證的 RT Plan

## 0. 環境準備

需要的套件：`numpy`、`pillow`、`matplotlib`、`scipy`、`pydicom`。

In [ ]:
# !pip install numpy pillow matplotlib scipy pydicom

import copy
import datetime
import logging
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

import numpy as np
import pydicom
from PIL import Image
import matplotlib.pyplot as plt

from pydicom.dataset import Dataset
from pydicom.sequence import Sequence
from pydicom.uid import ExplicitVRLittleEndian, generate_uid

logging.basicConfig(level=logging.INFO, format="%(message)s")
np.set_printoptions(precision=2, suppress=True, linewidth=140)
plt.rcParams["figure.dpi"] = 100
print("pydicom", pydicom.__version__)

## 0.1 參數設定

所有可調參數集中在這裡，方便教學實驗時修改觀察效果。

In [ ]:
@dataclass
class Params:
    # ---- 輸入 / 輸出 ----
    rp_path: str = "demo.dcm"
    image_path: str = "target_fluence.png"
    out_sliding: str = "output_sliding.dcm"
    out_stepshoot: str = "output_stepshoot.dcm"

    # ---- 照野幾何 (mm, isocenter 平面) ----
    field_x_mm: float = 100.0
    field_y_mm: float = 100.0
    field_center_x_mm: float = 0.0
    field_center_y_mm: float = 0.0
    pixel_mm: float = 2.0                # 兩種演算法共用，方便比較

    # ---- 圖像 → fluence ----
    invert: bool = True                  # 黑=高 fluence
    gamma: float = 1.0

    # ---- 出束 / 機器限制 ----
    total_mu: float = 100.0
    dose_rate_mu_min: Optional[float] = None  # None → 讀 RP，若無則用 600
    max_leaf_speed_mm_s: float = 25.0
    leaf_speed_margin: float = 0.9
    min_leaf_gap_mm: float = 1.0

    # ---- Sliding window ----
    cp_candidates: Tuple[int, ...] = (50, 100, 200, 400)
    target_nrmse_pct: float = 2.0

    # ---- Step-and-shoot ----
    max_segments: int = 60
    min_segment_mu: float = 1.0
    max_levels: int = 32
    refine_weights: bool = True

    # ---- RP 處理 ----
    beam_index: Optional[int] = None
    new_study_uid: bool = False

p = Params()
print(p)

---
# 1. 讀取輸入：RT Plan (MLC 幾何) 與 fluence 圖像

從 `demo.dcm` 讀出：
- MLC 類型（`MLCX` / `MLCY`）
- 葉片邊界 `LeafPositionBoundaries`
- 有哪些 jaw（`X` / `Y` / `ASYMX` / `ASYMY`）
- 原始 dose rate（`DoseRateSet`）

In [ ]:
@dataclass
class MlcInfo:
    device_type: str                 # "MLCX" 或 "MLCY"
    boundaries: np.ndarray           # (n_pairs+1,)
    jaw_types: List[str] = field(default_factory=list)

    @property
    def n_pairs(self):
        return len(self.boundaries) - 1


MLC_TYPES = ("MLCX", "MLCY")


def _bld_type(bld):
    return str(bld.RTBeamLimitingDeviceType).upper()


def find_beam(rp, beam_index=None):
    seq = rp.BeamSequence
    if beam_index is not None:
        return beam_index, seq[beam_index]
    for i, beam in enumerate(seq):
        if any(_bld_type(b) in MLC_TYPES for b in beam.BeamLimitingDeviceSequence):
            return i, beam
    raise ValueError("RP 裡沒有任何帶 MLC 的 beam")


def read_mlc_info(beam) -> MlcInfo:
    mlc_type, boundaries, jaws = None, None, []
    for bld in beam.BeamLimitingDeviceSequence:
        t = _bld_type(bld)
        if t in MLC_TYPES:
            b = getattr(bld, "LeafPositionBoundaries", None)
            if b is None:
                raise ValueError(f"{t} 缺少 LeafPositionBoundaries")
            mlc_type = t
            boundaries = np.array([float(v) for v in b])
        else:
            jaws.append(t)
    if mlc_type is None:
        raise ValueError("此 beam 沒有 MLC 裝置")
    return MlcInfo(mlc_type, boundaries, jaws)


def original_dose_rate(beam):
    try:
        v = getattr(beam.ControlPointSequence[0], "DoseRateSet", None)
        return float(v) if v is not None else None
    except Exception:
        return None


rp = pydicom.dcmread(p.rp_path)
beam_index, beam = find_beam(rp, p.beam_index)
mlc = read_mlc_info(beam)
dose_rate = p.dose_rate_mu_min or original_dose_rate(beam) or 600.0

print(f"使用 beam #{beam.BeamNumber} '{getattr(beam, 'BeamName', '')}'")
print(f"MLC 類型 = {mlc.device_type}，{mlc.n_pairs} 對葉片")
print(f"jaw = {mlc.jaw_types}")
print(f"dose rate = {dose_rate:.0f} MU/min")
print(f"葉片邊界 (mm): {mlc.boundaries[:3]} ... {mlc.boundaries[-3:]}")

---
# 2. 圖像 → 目標 Fluence Map

把 `target_fluence.png` 的灰階值當作 fluence `I ∈ [0,1]`，重採樣到照野解析度。

- `invert=True` 時，黑 → 高 fluence
- 圖像上方 → isocenter 的 +Y 方向

In [ ]:
@dataclass
class FluenceMap:
    data: np.ndarray        # (ny, nx)，row→+Y，col→+X
    x_edges: np.ndarray
    y_edges: np.ndarray

    @property
    def x_centers(self):
        return 0.5 * (self.x_edges[:-1] + self.x_edges[1:])

    @property
    def y_centers(self):
        return 0.5 * (self.y_edges[:-1] + self.y_edges[1:])


def load_fluence(p: Params) -> FluenceMap:
    nx = int(round(p.field_x_mm / p.pixel_mm))
    ny = int(round(p.field_y_mm / p.pixel_mm))

    img = Image.open(p.image_path)
    if img.mode in ("RGBA", "LA") or "transparency" in img.info:
        rgba = img.convert("RGBA")
        bg = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
        img = Image.alpha_composite(bg, rgba)
    img = img.convert("L").resize((nx, ny), Image.LANCZOS)

    arr = np.asarray(img, dtype=np.float64) / 255.0
    f = (1.0 - arr) if p.invert else arr
    f = np.clip(f, 0.0, 1.0) ** p.gamma
    f = f[::-1, :]                       # 圖像上 → +Y
    if f.max() <= 1e-9:
        raise ValueError("fluences 全為 0（invert 設定正確？）")

    x0 = p.field_center_x_mm - p.field_x_mm / 2
    y0 = p.field_center_y_mm - p.field_y_mm / 2
    return FluenceMap(
        data=np.ascontiguousarray(f),
        x_edges=x0 + np.arange(nx + 1) * (p.field_x_mm / nx),
        y_edges=y0 + np.arange(ny + 1) * (p.field_y_mm / ny),
    )


fm = load_fluence(p)
print(f"fluence map 形狀 = {fm.data.shape}，值域 = [{fm.data.min():.2f}, {fm.data.max():.2f}]")

plt.figure(figsize=(5, 4))
plt.imshow(fm.data, origin="lower", cmap="gray_r",
           extent=[fm.x_edges[0], fm.x_edges[-1], fm.y_edges[0], fm.y_edges[-1]])
plt.colorbar(label="fluence (0–1)")
plt.xlabel("X (mm)"); plt.ylabel("Y (mm)")
plt.title("目標 Fluence Map")
plt.show()

## 2.1 抽取每片葉片的 1D 目標 fluence

對每片葉片所在 Y 範圍做**面積加權平均**，得到 `I_r(x)`。

與照野無重疊的葉片 `active=False`（不出束，停在關閉位置）。

In [ ]:
def leaf_row_profiles(data, row_edges, boundaries):
    """(ny,nx) fluence → (n_leaves, nx) 每片葉片的 1D 平均 profile。"""
    n_rows, n_cols = data.shape
    dy = row_edges[1] - row_edges[0]
    csum = np.vstack([np.zeros((1, n_cols)), np.cumsum(data, axis=0)])

    lo = np.clip(boundaries[:-1], row_edges[0], row_edges[-1])
    hi = np.clip(boundaries[1:], row_edges[0], row_edges[-1])

    def cum_at(y):
        pos = (y - row_edges[0]) / dy
        i = np.clip(np.floor(pos).astype(int), 0, n_rows - 1)
        frac = (pos - i)[:, None]
        return csum[i] + frac * (csum[i + 1] - csum[i])

    area = cum_at(hi) - cum_at(lo)
    width = (hi - lo) / dy
    covered = width > 1e-6
    profiles = np.zeros_like(area)
    profiles[covered] = area[covered] / width[covered, None]
    return profiles, covered


# MLCX 時葉片沿 X 移動、列沿 Y；MLCY 時是轉置
if mlc.device_type == "MLCX":
    data, row_edges = fm.data, fm.y_edges
    row_centers, travel = fm.y_centers, fm.x_centers
else:
    data, row_edges = fm.data.T, fm.x_edges
    row_centers, travel = fm.x_centers, fm.y_centers

profiles, covered = leaf_row_profiles(data, row_edges, mlc.boundaries)
active = covered & (profiles.max(axis=1) > 1e-6)
print(f"需出束的葉片對 = {active.sum()} / {len(active)}")

fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
for ax, r in zip(axes, [0, mlc.n_pairs // 2, mlc.n_pairs - 1]):
    ax.plot(travel, profiles[r], color="C0")
    ax.set_ylabel(f"row {r}\nactive={active[r]}")
    ax.grid(alpha=0.3)
axes[-1].set_xlabel("travel (mm)")
axes[0].set_title("每片葉片的 1D 目標 fluence")
plt.tight_layout(); plt.show()

---
# 3. Sliding Window 演算法

### 模型
每個 leaf pair 由左往右（+travel）掃過照野：
- **B 排（leading edge，右葉）** 領先
- **A 排（trailing edge，左葉）** 跟隨
- 位置 x 的 fluence `= T_A(x) − T_B(x)`（累積 MU 差）
- 速度受限：`dT/dx ≥ g`，`g = dose_rate / (leaf_speed × margin)`

### 最佳解
對相鄰取樣 `ΔI = I[j] − I[j-1]`：
```
T_B[j] = T_B[j-1] + g·dx + max(0, −ΔI)   # fluence 下降時 B 變慢
T_A[j] = T_A[j-1] + g·dx + max(0, +ΔI)   # fluence 上升時 A 變慢
```
→ 滿足 `T_A − T_B = I`，且至少一片葉片全速前進。

In [ ]:
def mu_per_mm(dose_rate_mu_min, max_speed_mm_s, margin=0.9):
    """葉片以 margin×最大速度前進時，每 1 mm 對應的 MU。"""
    return (dose_rate_mu_min / 60.0) / (max_speed_mm_s * margin)


g = mu_per_mm(dose_rate, p.max_leaf_speed_mm_s, p.leaf_speed_margin)
print(f"g = {g:.4f} MU/mm")
print(f"→ 光是掃過 {p.field_x_mm:.0f} mm 照野就要 {g * p.field_x_mm:.1f} MU")

### 3.1 計算每片葉片的累積 MU 軌跡 `T_A(x)` 與 `T_B(x)`

關鍵：整個 beam 的總 MU 由**最慢的那一列**決定，所以我們反過來解「fluence 縮放係數 s」讓 beam 總 MU = `total_mu`。

In [ ]:
@dataclass
class Trajectories:
    x: np.ndarray          # (npts,)
    t_a: np.ndarray        # (n_leaves, npts)
    t_b: np.ndarray        # (n_leaves, npts)
    active: np.ndarray
    scale: float           # 正規化 fluence → MU
    total_mu: float
    g: float


def compute_trajectories(profiles, active, x_centers, total_mu, g, min_gap_mm):
    dx = float(x_centers[1] - x_centers[0])
    pad_n = int(np.ceil(min_gap_mm / dx)) + 1
    x = np.concatenate([
        x_centers[0] - dx * np.arange(pad_n, 0, -1),
        x_centers,
        x_centers[-1] + dx * np.arange(1, pad_n + 1),
    ])
    P = np.pad(profiles, ((0, 0), (pad_n, pad_n)))
    P[~active] = 0.0

    d = np.diff(P, axis=1)
    rise = np.maximum(d, 0.0).sum(axis=1)
    travel_mu = g * (x[-1] - x[0])
    if total_mu <= travel_mu * 1.001:
        raise ValueError(
            f"total_mu={total_mu:.1f} MU 太小：光是掃過照野就要 {travel_mu:.1f} MU。")
    scale = (total_mu - travel_mu) / rise[active].max()

    step = g * dx
    zeros = np.zeros((P.shape[0], 1))
    t_a = np.concatenate([zeros, np.cumsum(step + scale * np.maximum(d, 0.0), axis=1)], axis=1)
    t_b = np.concatenate([zeros, np.cumsum(step + scale * np.maximum(-d, 0.0), axis=1)], axis=1)
    return Trajectories(x=x, t_a=t_a, t_b=t_b, active=active.copy(),
                        scale=float(scale), total_mu=float(total_mu), g=float(g))


traj = compute_trajectories(profiles, active, travel, p.total_mu, g, p.min_leaf_gap_mm)
print(f"縮放係數 scale = {traj.scale:.3f}（fluence 1.0 → {traj.scale:.1f} MU）")
print(f"beam 總 MU = {traj.total_mu:.1f}")

### 3.2 視覺化軌跡（教學重點）

觀察 `T_A − T_B` 如何在每個位置 x 上剛好等於目標 fluence。

In [ ]:
r = int(np.argmax(active))
target_rows = profiles * traj.scale
target_rows[~active] = 0.0

fig, ax = plt.subplots(1, 2, figsize=(13, 4))

ax[0].fill_between(travel, 0, target_rows[r], alpha=0.4, color="C0")
ax[0].plot(travel, target_rows[r], "C0", label=f"目標 fluence × {traj.scale:.1f} MU")
ax[0].set_xlabel("travel (mm)"); ax[0].set_ylabel("MU")
ax[0].set_title(f"Row {r} 目標 fluence")
ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(traj.x, traj.t_a[r], "C3", label="T_A(x) 左葉")
ax[1].plot(traj.x, traj.t_b[r], "C2", label="T_B(x) 右葉")
ax[1].fill_between(traj.x, traj.t_b[r], traj.t_a[r], color="C0", alpha=0.3,
                   label="T_A − T_B = fluence")
ax[1].set_xlabel("travel (mm)"); ax[1].set_ylabel("累積 MU")
ax[1].set_title("兩葉片的累積 MU 軌跡")
ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 3.3 在每個 control point 取樣葉片位置

把 CP 的累積 MU `t_mu` 對到軌跡上，得到 `A[:, r]` 與 `B[:, r]`。

In [ ]:
def sample_leaf_positions(traj, t_mu, min_gap_mm, decimals=2):
    n = traj.t_a.shape[0]
    A = np.empty((len(t_mu), n))
    B = np.empty((len(t_mu), n))
    for r in range(n):
        if traj.active[r]:
            A[:, r] = np.interp(t_mu, traj.t_a[r], traj.x)
            B[:, r] = np.interp(t_mu, traj.t_b[r], traj.x)
        else:
            A[:, r] = B[:, r] = traj.x[0]

    q = 10 ** decimals
    gap = np.ceil(min_gap_mm * q) / q
    A, B = np.round(A, decimals), np.round(B, decimals)
    A = np.round(np.minimum(A, B - gap), decimals)
    return A, B


n_cp = 100
t_mu = np.linspace(0.0, traj.total_mu, n_cp)
A_slide, B_slide = sample_leaf_positions(traj, t_mu, p.min_leaf_gap_mm)
print(f"A shape = {A_slide.shape}  (n_cp={n_cp}, n_leaves={mlc.n_pairs})")
print(f"CP0  A[:3]={A_slide[0,:3]}, B[:3]={B_slide[0,:3]}")
print(f"CP99 A[:3]={A_slide[-1,:3]}, B[:3]={B_slide[-1,:3]}")

### 3.4 正向模擬與誤差

由 CP 序列模擬 fluence（CP 之間葉片線性內插），與目標比對。

In [ ]:
def simulate_rows(A, B, t_mu, x_centers):
    n = A.shape[1]
    out = np.zeros((n, len(x_centers)))
    for r in range(n):
        if np.ptp(A[:, r]) == 0 and np.ptp(B[:, r]) == 0:
            continue
        ta = np.interp(x_centers, A[:, r], t_mu)
        tb = np.interp(x_centers, B[:, r], t_mu)
        out[r] = np.maximum(ta - tb, 0.0)
    return out


def compare(target_mu, sim_mu, active):
    diff = (sim_mu - target_mu)[active]
    peak = target_mu[active].max()
    return {
        "nrmse_pct": float(np.sqrt(np.mean(diff ** 2)) / peak * 100),
        "max_abs_err_pct": float(np.abs(diff).max() / peak * 100),
    }


sim_slide = simulate_rows(A_slide, B_slide, t_mu, travel)
m_slide = compare(target_rows, sim_slide, active)
print(f"CP={n_cp}: NRMSE = {m_slide['nrmse_pct']:.2f}%，最大誤差 = {m_slide['max_abs_err_pct']:.2f}%")

### 3.5 CP 加密實驗：CP 越多誤差越小？

In [ ]:
print("CP 數 vs NRMSE：")
cand_results = []
best = None
for n in p.cp_candidates:
    t = np.linspace(0.0, traj.total_mu, n)
    A_, B_ = sample_leaf_positions(traj, t, p.min_leaf_gap_mm)
    s = simulate_rows(A_, B_, t, travel)
    mm = compare(target_rows, s, active)
    cand_results.append((n, mm["nrmse_pct"]))
    print(f"  CP={n:4d}  NRMSE={mm['nrmse_pct']:5.2f}%  最大誤差={mm['max_abs_err_pct']:5.2f}%")
    if best is None or mm["nrmse_pct"] < best[0]["nrmse_pct"]:
        best = (mm, t, A_, B_, s)
    if mm["nrmse_pct"] <= p.target_nrmse_pct:
        break

m_slide, t_mu, A_slide, B_slide, sim_slide = best
print(f"\n最終採用：{len(t_mu)} CP，NRMSE={m_slide['nrmse_pct']:.2f}%")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot([r[0] for r in cand_results], [r[1] for r in cand_results], "o-")
ax.set_xscale("log")
ax.set_xlabel("control point 數"); ax.set_ylabel("NRMSE (%)")
ax.axhline(p.target_nrmse_pct, color="r", ls="--", label=f"目標 {p.target_nrmse_pct}%")
ax.legend(); ax.grid(alpha=0.3, which="both")
ax.set_title("Sliding window：CP 數對誤差的影響")
plt.tight_layout(); plt.show()

In [ ]:
# 準備寫回 DICOM 用的 Delivery 物件（sliding）
weights_slide = np.round(t_mu / traj.total_mu, 6)
weights_slide[0], weights_slide[-1] = 0.0, 1.0

print(f"weights[0]={weights_slide[0]}, weights[-1]={weights_slide[-1]}")
print(f"n_cp = {len(weights_slide)}")

---
# 4. Step-and-Shoot 演算法

### 核心概念
1. 每列 1D fluence **量化成整數層級** `I[j] ∈ {0..n_levels}`
2. **分解**：I 上升處產生「左邊界」、下降處產生「右邊界」
   - 上升 ΔI 就重複 ΔI 次（每個 segment 各開一層）
   - 排序好的左右邊界**依序配對**，第 k 對就是第 k 個 segment 的開口
   - **Σ_k 1[L_k ≤ x < R_k] = I(x)** 精確成立（每 segment 權重 = 1 層）
3. 每個 segment 用兩個 CP（位置相同、累積權重不同），`BeamType = STATIC`

### 4.1 一維分解示範（教學重點）

In [ ]:
def decompose_row(levels):
    """回傳 (left_edge_idx, right_edge_idx)。"""
    nx = len(levels)
    d = np.diff(np.concatenate([[0], levels, [0]]))  # 兩端補 0
    edges = np.arange(nx + 1)
    return np.repeat(edges, np.maximum(d, 0)), \
           np.repeat(edges, np.maximum(-d, 0))


demo = np.array([0, 1, 2, 3, 2, 1, 0])
L, R = decompose_row(demo)
print("levels  =", demo)
print("左邊界  =", L)
print("右邊界  =", R)
print(f"→ 配對後得到 {len(L)} 個 segment")
acc = np.zeros_like(demo, dtype=int)
for k, (l, r) in enumerate(zip(L, R)):
    seg = np.zeros_like(demo, dtype=int)
    seg[l:r] = 1
    acc += seg
    print(f"  seg {k}: [{l}, {r})  貢獻 {seg}")
print(f"Σ segments = {acc}  （與 levels 相符：{np.array_equal(acc, demo)}）")

### 4.2 對所有葉片做分解，並依 segment 序號排列

In [ ]:
def quantize_rows(profiles, active, n_levels):
    levels = np.rint(profiles * n_levels).astype(int)
    levels[~active] = 0
    return levels


def build_segments(levels, x_edges, min_gap_mm):
    """回傳 A, B (M, n_leaves)、is_open (M, n_leaves)。"""
    n_leaves = levels.shape[0]
    rows = [decompose_row(levels[r]) for r in range(n_leaves)]
    M = max(len(L) for L, _ in rows)
    A = np.full((M, n_leaves), x_edges[0] - min_gap_mm)
    B = np.full((M, n_leaves), x_edges[0])
    is_open = np.zeros((M, n_leaves), dtype=bool)
    for r, (L, R) in enumerate(rows):
        k = len(L)
        if k == 0:
            continue
        a, b = x_edges[L], x_edges[R]
        b = np.maximum(b, a + min_gap_mm)
        A[:k, r], B[:k, r], is_open[:k, r] = a, b, True
    return np.round(A, 2), np.round(B, 2), is_open


def segment_footprints(A, B, x_edges):
    """每個 segment 對每個像素的開啟比例，形狀 (M, n_leaves, nx)。"""
    e0, e1 = x_edges[:-1], x_edges[1:]
    dx = e1[0] - e0[0]
    ov = np.minimum(B[:, :, None], e1) - np.maximum(A[:, :, None], e0)
    return np.clip(ov, 0.0, dx) / dx


# 建立 travel 方向像素邊界
dx = float(travel[1] - travel[0])
x_edges = travel[0] - dx / 2 + np.arange(len(travel) + 1) * dx

print("層數 → segment 數：")
for n in [4, 8, 16, 32]:
    lv = quantize_rows(profiles, active, n)
    A_, B_, _ = build_segments(lv, x_edges, p.min_leaf_gap_mm)
    print(f"  n_levels={n:3d}  →  M={A_.shape[0]:4d} segments")

### 4.3 自動挑選層數

選最大 `n_levels` 使得 segment 數 `≤ max_segments` 且每個 segment 的 MU `≥ min_segment_mu`。

In [ ]:
chosen = None
for n in range(p.max_levels, 1, -1):
    lv = quantize_rows(profiles, active, n)
    A_, B_, is_open_ = build_segments(lv, x_edges, p.min_leaf_gap_mm)
    M = A_.shape[0]
    if M <= p.max_segments and p.total_mu / M >= p.min_segment_mu:
        chosen = (n, lv, A_, B_, is_open_, M)
        break

if chosen is None:
    raise ValueError("找不到符合條件的層數，放寬 max_segments 或增加 total_mu")

n_levels, levels, A_sas, B_sas, is_open_sas, M = chosen
print(f"選定：{n_levels} 層，{M} 個 segment，等權重約 {p.total_mu/M:.2f} MU/seg")

### 4.4 視覺化 segment（前 8 個）

In [ ]:
def plot_segments(A, B, is_open, n_show=8):
    M = A.shape[0]
    n = min(M, n_show)
    fig, axes = plt.subplots(n, 1, figsize=(10, 1.1 * n), sharex=True)
    if n == 1:
        axes = [axes]
    for k in range(n):
        ax = axes[k]
        for r in range(A.shape[1]):
            if is_open[k, r]:
                ax.plot([A[k, r], B[k, r]], [r, r], "C0-", lw=3)
        ax.set_ylim(-1, A.shape[1])
        ax.set_ylabel(f"seg {k}")
        ax.grid(alpha=0.2)
    axes[-1].set_xlabel("travel (mm)")
    plt.tight_layout(); plt.show()


plot_segments(A_sas, B_sas, is_open_sas, n_show=8)
print(f"（只顯示前 8 個，共 {M} 個 segment）")

### 4.5 權重擬合：等權重 vs NNLS

- **等權重**：每個 segment `MU = total_mu / M`
- **NNLS**：最小化模擬與目標的差異，並剔除 MU 太小的 segment

In [ ]:
F_sas = segment_footprints(A_sas, B_sas, x_edges)
w_equal = np.full(M, p.total_mu / M)
target0 = profiles * (n_levels * p.total_mu / M)
target0[~active] = 0.0

sim_eq = np.tensordot(w_equal, F_sas, axes=1)
diff_eq = (sim_eq - target0)[active]
nrmse_eq = np.sqrt(np.mean(diff_eq ** 2)) / target0[active].max() * 100
print(f"等權重 NRMSE = {nrmse_eq:.2f}%")

def _nnls_weights(F, target, active, min_mu):
    from scipy.optimize import nnls
    keep = np.arange(F.shape[0])
    b = target[active].ravel()
    w = None
    low = None
    for _ in range(5):
        Amat = F[keep][:, active, :].reshape(len(keep), -1).T
        w, _ = nnls(Amat, b, maxiter=50 * len(keep))
        low = w < min_mu
        if not low.any() or low.all():
            break
        keep = keep[~low]
    if w is None or (low is not None and low.all()):
        return None, None
    return keep, w


if p.refine_weights:
    keep, w_nn = _nnls_weights(F_sas, target0, active, p.min_segment_mu)
    if keep is not None:
        A_sas, B_sas, is_open_sas = A_sas[keep], B_sas[keep], is_open_sas[keep]
        F_sas = F_sas[keep]
        w_nn = w_nn * (p.total_mu / w_nn.sum())
        sim_nn = np.tensordot(w_nn, F_sas, axes=1)
        diff_nn = (sim_nn - target0)[active]
        nrmse_nn = np.sqrt(np.mean(diff_nn ** 2)) / target0[active].max() * 100
        print(f"NNLS：保留 {len(keep)}/{M} 個 segment，最小 MU={w_nn.min():.2f}，NRMSE={nrmse_nn:.2f}%")
        w_sas, sim_sas, M = w_nn, sim_nn, len(keep)
    else:
        print("NNLS 全低於 min_mu，使用等權重")
        w_sas, sim_sas = w_equal, sim_eq
else:
    w_sas, sim_sas = w_equal, sim_eq

In [ ]:
# 準備寫回 DICOM 用的 CP 序列（每個 segment 兩個 CP）
n_seg = len(w_sas)
cum = np.round(np.cumsum(w_sas) / w_sas.sum(), 6)
cum[-1] = 1.0
prev = np.concatenate([[0.0], cum[:-1]])
weights_sas = np.empty(2 * n_seg)
weights_sas[0::2], weights_sas[1::2] = prev, cum
A_sas_cp = np.repeat(A_sas, 2, axis=0)
B_sas_cp = np.repeat(B_sas, 2, axis=0)
print(f"stepshoot: {n_seg} segments → {len(weights_sas)} CP")
print(f"weights[0:4] = {weights_sas[:4]}")

---
# 5. 兩種演算法並排比較

In [ ]:
def rows_to_grid(rows_mu, boundaries, row_centers):
    idx = np.clip(np.searchsorted(boundaries, row_centers, side="right") - 1,
                  0, rows_mu.shape[0] - 1)
    return rows_mu[idx]


# 把「每片葉片的 1D 結果」展開回 fluence 網格 (ny, nx)
delivered_slide = rows_to_grid(sim_slide, mlc.boundaries, row_centers)
delivered_sas   = rows_to_grid(sim_sas,   mlc.boundaries, row_centers)
if mlc.device_type == "MLCY":
    # MLCY 時 row_centers 是 x 方向，travel 是 y 方向 → 結果形狀為 (nx, ny)，需轉置
    delivered_slide = delivered_slide.T
    delivered_sas   = delivered_sas.T

target_mu = fm.data * traj.scale
assert delivered_slide.shape == target_mu.shape, \
    f"形狀不符：{delivered_slide.shape} vs {target_mu.shape}"
assert delivered_sas.shape == target_mu.shape

vmax = float(target_mu.max())
ext  = [fm.x_edges[0], fm.x_edges[-1], fm.y_edges[0], fm.y_edges[-1]]

# ---- 修正解包：用 zip 把 ax 和 imgs 對齊 ----
imgs = [
    (target_mu,      "Target (MU)",
     dict(cmap="gray_r", vmin=0, vmax=vmax)),
    (delivered_slide, f"Sliding window\n({len(weights_slide)} CP)",
     dict(cmap="gray_r", vmin=0, vmax=vmax)),
    (delivered_sas,   f"Step-and-shoot\n({n_seg} seg)",
     dict(cmap="gray_r", vmin=0, vmax=vmax)),
    (delivered_slide - target_mu, "Sliding − Target",
     dict(cmap="coolwarm", vmin=-0.15*vmax, vmax=0.15*vmax)),
]

fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for a, (img, title, kw) in zip(ax, imgs):
    im = a.imshow(img, origin="lower", extent=ext, **kw)
    a.set_title(title); a.set_xlabel("X (mm)"); a.set_ylabel("Y (mm)")
    fig.colorbar(im, ax=a, fraction=0.046)

# 只看 fluence 形狀（normalize 到同一峰值）
target_shape   = target_mu / target_mu.max()
slide_shape    = delivered_slide / delivered_slide.max()
sas_shape      = delivered_sas / delivered_sas.max()
print(f"形狀 NRMSE（sliding） = {np.sqrt(np.mean((slide_shape-target_shape)**2))*100:.2f}%")
print(f"形狀 NRMSE（sas）     = {np.sqrt(np.mean((sas_shape-target_shape)**2))*100:.2f}%")

nr_slide = np.sqrt(np.mean((delivered_slide - target_mu) ** 2)) / vmax * 100
nr_sas   = np.sqrt(np.mean((delivered_sas   - target_mu) ** 2)) / vmax * 100
fig.suptitle(f"Sliding NRMSE={nr_slide:.2f}%  |  Step-and-shoot NRMSE={nr_sas:.2f}%")
fig.tight_layout(); plt.show()

---
# 6. 寫回 DICOM

以 `demo.dcm` 為模板：
- 深拷貝，不動到原檔
- 只保留目標 beam（`BeamSequence` 只放一支）
- `FractionGroupSequence` 只保留指向這支 beam 的 reference
- 更新 `FractionGroup` 的 `BeamMeterset = total_mu`
- 更新 `RTPlanLabel`、`SOPInstanceUID`、日期，狀態設為 `UNAPPROVED`
- `Beam.ControlPointSequence` 換成我們算出的 CP 序列

In [ ]:
JAW_X = {"X", "ASYMX"}
JAW_Y = {"Y", "ASYMY"}


def _r(v, nd):
    return float(round(float(v), nd))


def _bld_item(dev_type, values):
    d = Dataset()
    d.RTBeamLimitingDeviceType = dev_type
    d.LeafJawPositions = [_r(v, 2) for v in values]
    return d


def _fill_cp0_devices(cp0, mlc, mlc_values, jaw_x, jaw_y):
    seq = getattr(cp0, "BeamLimitingDevicePositionSequence", None)
    if seq is None:
        seq = Sequence()
        cp0.BeamLimitingDevicePositionSequence = seq
    seen = set()
    for item in seq:
        t = str(item.RTBeamLimitingDeviceType).upper()
        seen.add(t)
        if t == mlc.device_type:
            item.LeafJawPositions = [_r(v, 2) for v in mlc_values]
        elif t in JAW_X:
            item.LeafJawPositions = [_r(v, 2) for v in jaw_x]
        elif t in JAW_Y:
            item.LeafJawPositions = [_r(v, 2) for v in jaw_y]
    if mlc.device_type not in seen:
        seq.append(_bld_item(mlc.device_type, mlc_values))
    for t in mlc.jaw_types:
        if t not in seen:
            if t in JAW_X:
                seq.append(_bld_item(t, jaw_x))
            elif t in JAW_Y:
                seq.append(_bld_item(t, jaw_y))


def build_plan(rp_template, beam_index, mlc, A, B, weights, total_mu, beam_type, plan_label):
    """A, B: (n_cp, n_leaves)；weights: (n_cp,) 0→1。"""
    new = copy.deepcopy(rp_template)
    beam = new.BeamSequence[beam_index]
    beam_number = int(beam.BeamNumber)
    new.BeamSequence = Sequence([beam])

    # ---- FractionGroup ----
    groups = []
    for fg in new.FractionGroupSequence:
        keep = [rb for rb in fg.ReferencedBeamSequence
                if int(rb.ReferencedBeamNumber) == beam_number]
        if not keep:
            continue
        for rb in keep:
            rb.BeamMeterset = _r(total_mu, 4)
            for kw in ("BeamDose", "BeamDoseSpecificationPoint"):
                if hasattr(rb, kw):
                    delattr(rb, kw)
        fg.ReferencedBeamSequence = Sequence(keep)
        fg.NumberOfBeams = len(keep)
        groups.append(fg)
    if not groups:
        raise ValueError(f"FractionGroup 找不到 BeamNumber={beam_number} 的 reference")
    new.FractionGroupSequence = Sequence(groups)

    # ---- Plan 層級 ----
    now = datetime.datetime.now()
    new.RTPlanDate = now.strftime("%Y%m%d")
    new.RTPlanTime = now.strftime("%H%M%S")
    new.RTPlanLabel = plan_label[:16]
    new.SOPInstanceUID = generate_uid()
    new.SeriesInstanceUID = generate_uid()
    if p.new_study_uid:
        new.StudyInstanceUID = generate_uid()
    new.ApprovalStatus = "UNAPPROVED"
    for kw in ("ReviewDate", "ReviewTime", "ReviewerName"):
        if hasattr(new, kw):
            delattr(new, kw)
    new.file_meta.MediaStorageSOPInstanceUID = new.SOPInstanceUID
    new.file_meta.TransferSyntaxUID = ExplicitVRLittleEndian

    # ---- Beam：CP 序列 ----
    orig_cps = beam.ControlPointSequence
    cp0 = copy.deepcopy(orig_cps[0])
    last_ref = getattr(orig_cps[-1], "ReferencedDoseReferenceSequence", None)
    last_ref = copy.deepcopy(last_ref) if last_ref is not None else None

    n_cp = len(weights)
    beam.BeamType = beam_type
    beam.NumberOfControlPoints = n_cp
    beam.FinalCumulativeMetersetWeight = 1.0

    jaw_x = [p.field_center_x_mm - p.field_x_mm / 2,
             p.field_center_x_mm + p.field_x_mm / 2]
    jaw_y = [p.field_center_y_mm - p.field_y_mm / 2,
             p.field_center_y_mm + p.field_y_mm / 2]

    cp0.ControlPointIndex = 0
    cp0.CumulativeMetersetWeight = 0.0
    _fill_cp0_devices(cp0, mlc, list(A[0]) + list(B[0]), jaw_x, jaw_y)
    cps = [cp0]
    for k in range(1, n_cp):
        cp = Dataset()
        cp.ControlPointIndex = k
        cp.CumulativeMetersetWeight = _r(weights[k], 6)
        cp.BeamLimitingDevicePositionSequence = Sequence(
            [_bld_item(mlc.device_type, list(A[k]) + list(B[k]))])
        cps.append(cp)
    if last_ref is not None and n_cp > 1:
        cps[-1].ReferencedDoseReferenceSequence = last_ref
    beam.ControlPointSequence = Sequence(cps)
    return new


def save_plan(ds, path):
    major = int(pydicom.__version__.split(".")[0])
    if major >= 3:
        ds.save_as(path, enforce_file_format=True)
    else:
        ds.is_little_endian = True
        ds.is_implicit_VR = False
        ds.save_as(path, write_like_original=False)


print("build_plan / save_plan 已定義")

### 6.1 寫出 Sliding Window 的 RT Plan

In [ ]:
rp_slide = build_plan(rp, beam_index, mlc,
                      A=A_slide, B=B_slide, weights=weights_slide,
                      total_mu=traj.total_mu, beam_type="DYNAMIC",
                      plan_label="MLC_SLIDING")
save_plan(rp_slide, p.out_sliding)
print(f"已寫出 {p.out_sliding}")
print(f"  BeamType = {rp_slide.BeamSequence[0].BeamType}")
print(f"  NumberOfControlPoints = {rp_slide.BeamSequence[0].NumberOfControlPoints}")
print(f"  RTPlanLabel = {rp_slide.RTPlanLabel}")
print(f"  BeamMeterset = {rp_slide.FractionGroupSequence[0].ReferencedBeamSequence[0].BeamMeterset}")

### 6.2 寫出 Step-and-Shoot 的 RT Plan

In [ ]:
rp_sas = build_plan(rp, beam_index, mlc,
                    A=A_sas_cp, B=B_sas_cp, weights=weights_sas,
                    total_mu=p.total_mu, beam_type="STATIC",
                    plan_label="MLC_STEP")
save_plan(rp_sas, p.out_stepshoot)
print(f"已寫出 {p.out_stepshoot}")
print(f"  BeamType = {rp_sas.BeamSequence[0].BeamType}")
print(f"  NumberOfControlPoints = {rp_sas.BeamSequence[0].NumberOfControlPoints}")
print(f"  RTPlanLabel = {rp_sas.RTPlanLabel}")
print(f"  BeamMeterset = {rp_sas.FractionGroupSequence[0].ReferencedBeamSequence[0].BeamMeterset}")

### 6.3 回讀驗證

把寫出的 DICOM 讀回來，確認 CP 結構正確。

In [ ]:
for path in (p.out_sliding, p.out_stepshoot):
    ds = pydicom.dcmread(path)
    b = ds.BeamSequence[0]
    print(f"\n=== {path} ===")
    print(f"BeamType={b.BeamType}   CP數={b.NumberOfControlPoints}   "
          f"FinalCumWeight={b.FinalCumulativeMetersetWeight}")
    cps = b.ControlPointSequence
    print(f"  cp[0]  Index={cps[0].ControlPointIndex}  CumW={cps[0].CumulativeMetersetWeight}")
    print(f"  cp[1]  Index={cps[1].ControlPointIndex}  CumW={cps[1].CumulativeMetersetWeight}")
    print(f"  cp[-1] Index={cps[-1].ControlPointIndex} CumW={cps[-1].CumulativeMetersetWeight}")
    mlc_item = [it for it in cps[0].BeamLimitingDevicePositionSequence
                if str(it.RTBeamLimitingDeviceType).upper() == mlc.device_type][0]
    print(f"  cp[0] {mlc.device_type} 位置前 4 個 = {mlc_item.LeafJawPositions[:4]}")

---
# 7. 總結：兩種演算法的關鍵差異

| 項目 | Sliding Window | Step-and-Shoot |
|---|---|---|
| `BeamType` | `DYNAMIC` | `STATIC` |
| 葉片運動 | 連續（CP 之間線性內插） | 只在 segment 邊界跳動 |
| 每個 CP 的葉片位置 | 都不同 | 兩兩成對（同位置） |
| CP 數量 | 可任意加密 | 固定為 `2 × segment 數` |
| 誤差來源 | 取樣不足（CP 加密可改善） | 量化層數（加層可改善） |
| MU 效率 | 較高 | 較低（segment 間 idle time） |
| 演算法核心 | `T_A − T_B = I`，累積 MU 軌跡 | 1D 層級分解 `Σ 1[L,R) = I` |

## 延伸思考題（給學生）

1. 若 `dose_rate_mu_min` 提高，sliding window 的誤差會如何變化？（提示：`g = dose_rate / leaf_speed`，`g↑` 表示需要更多 MU 才能掃過照野）
2. 若 `min_leaf_gap_mm` 增大，兩種演算法各受什麼影響？
3. Step-and-shoot 為什麼需要 NNLS 修正？只用等權重時誤差差多少？
4. `total_mu` 太小會發生什麼事？觀察 `compute_trajectories` 的錯誤訊息。
5. 把 `pixel_mm` 從 2.0 改成 1.0，兩種演算法的表現會怎麼變？再改回 4.0 觀察。
6. 為什麼 sliding window 至少要一片葉片全速移動？（提示：這是 optimality 條件，讓 beam 在時間上不浪費）

## 練習題（可修改 notebook 嘗試）

1. 把 `p.dose_rate_mu_min` 改為 300 再跑一次，比較兩種演算法的 NRMSE 與 MU。
2. 把 `p.max_segments` 從 60 改成 20，觀察 step-and-shoot 的誤差變化。
3. 換成 `p.invert = False`（白 = 高 fluence），觀察 fluence map 的變化。
4. 嘗試把 `p.min_leaf_gap_mm` 設為 3.0，觀察哪種演算法受影響更大。